# Fraud Detection Model

Training a Random Forest classifier to detect fraudulent transactions.
This notebook is designed to be ingested by `mlops-agents ingest`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_auc_score

## Generate Synthetic Data

5,000 transactions with 3% fraud rate. Features: amount, hour, merchant category, international flag, velocity, distance from home.

In [ ]:
np.random.seed(42)
n_samples = 5000
n_fraud = int(n_samples * 0.03)
n_legit = n_samples - n_fraud

# Legitimate transactions
legit = pd.DataFrame({
    'amount': np.random.lognormal(3.5, 1.0, n_legit),
    'hour': np.random.normal(14, 4, n_legit).clip(0, 23),
    'category': np.random.randint(0, 10, n_legit),
    'is_international': np.random.binomial(1, 0.05, n_legit),
    'velocity_1h': np.random.poisson(2, n_legit),
    'distance_from_home': np.random.exponential(10, n_legit),
    'is_fraud': 0
})

# Fraudulent transactions
fraud = pd.DataFrame({
    'amount': np.random.lognormal(5.5, 1.5, n_fraud),
    'hour': np.random.normal(3, 3, n_fraud).clip(0, 23),
    'category': np.random.randint(0, 10, n_fraud),
    'is_international': np.random.binomial(1, 0.4, n_fraud),
    'velocity_1h': np.random.poisson(8, n_fraud),
    'distance_from_home': np.random.exponential(100, n_fraud),
    'is_fraud': 1
})

df = pd.concat([legit, fraud]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Dataset: {len(df)} transactions, {df.is_fraud.mean():.1%} fraud rate')

In [ ]:
feature_cols = ['amount', 'hour', 'category', 'is_international', 'velocity_1h', 'distance_from_home']
X = df[feature_cols].values
y = df['is_fraud'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

## Train Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)
print('Model trained.')

## Evaluate

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'auc_roc': roc_auc_score(y_test, y_proba),
}

for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

## MLOps Manifest

This cell tells `mlops-agents ingest` which cells map to which pipeline stages.
No need to modify any working code above.

In [ ]:
# mlops: manifest
# cell 1: imports
# cell 3-4: data-loading
# cell 6: training
# cell 8: evaluation
# cell 9: metrics